<a href="https://colab.research.google.com/github/SalesforceAIResearch/SlackAgents/blob/main/docs/docs/examples/tools/tool_definition_from_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Tools from OpenAPI JSON File

This tutorial demonstrates how to use the OpenAPI JSON file to define the RESTful API tools for AI agents. Most of the available tools in the digital world are in the form of RESTful APIs with standard request formats, e.g., GET, POST, etc. While it is possible to write functions (see `tool_from_python_function.ipynb` or Pydantic classes `tool_from_pydantic.ipynb`), it is more convenient to use the OpenAPI JSON files to directly define the tools an agent can use in batch with a folder of JSON files.  


If you're opening this Notebook on colab, you will probably need to install `SlackAgents`. You can do this by running the following cell:

```python
!pip install slackagents
```

In [ ]:
!pip install slackagents

# Import the necessary modules
First, make sure you have the required modules imported:

In [1]:
from slackagents.tools.openapi_tool import OpenAPITool
from slackagents.tools.schema import AuthType
from slackagents.tools.base import ToolCall

## Prepare the OpenAPI specification
You need to have the OpenAPI specification for the API you want to interact with. This is usually a JSON file that describes the API endpoints, parameters, and responses. For this tutorial, we'll use the Chronicling America Title Search API as an example. 

[Chronicling America](https://www.postman.com/cs-demo/public-rest-apis/request/uxq3761/historical-american-newspapers) provides access to information about historic newspapers and select digitized newspaper pages. To encourage a wide range of potential uses, we designed several different views of the data we provide, all of which are publicly visible. Each uses common Web protocols, and access is not restricted in any way. You do not need to apply for a special key to use them. Together they make up an extensive application programming interface (API) which you can use to explore all of our data in many ways.

| Params | Value |
| --- | --- |
| terms | oakland |
| format | json |
| page | 1 |

```bash
curl --location 'https://chroniclingamerica.loc.gov/search/titles/results/?terms=oakland&format=json&page=5'
```

Most public APIs provide OpenAPI specifications automatically if they are deveoped using popular frameworks like [FastAPI](https://fastapi.tiangolo.com/tutorial/metadata/#openapi-url), Flask, etc, using Swagger or Redoc. Users can also easily use ChatGPT or other tools to generate OpenAPI specifications from the API documentation directly.

In [1]:
import json
with open("assets/chronicling.json") as f:
    openapi_spec = json.load(f)
print(openapi_spec)

{'openapi': '3.0.3', 'info': {'title': 'Chronicling America Title Search API', 'description': 'API for searching titles in the Chronicling America newspaper archive.', 'version': '1.0.0'}, 'servers': [{'url': 'https://chroniclingamerica.loc.gov'}], 'paths': {'/search/titles/results/': {'get': {'summary': 'Search newspaper titles', 'description': 'Retrieve newspaper titles that match specified search terms.', 'parameters': [{'name': 'terms', 'in': 'query', 'description': 'The search term or phrase for querying newspaper titles.', 'required': True, 'schema': {'type': 'string', 'example': 'oakland'}}, {'name': 'format', 'in': 'query', 'description': 'The format of the response.', 'required': True, 'schema': {'type': 'string', 'enum': ['json', 'xml'], 'example': 'json'}}, {'name': 'page', 'in': 'query', 'description': 'Page number for paginated results.', 'required': True, 'schema': {'type': 'integer', 'example': 5}}], 'responses': {'200': {'description': 'A list of newspaper titles that m

## Create an OpenAPITool instance
Now, create an instance of the `OpenAPITool` class using the OpenAPI specification:


In [6]:
tool = OpenAPITool(
    name="get_historical_american_newspapers",
    openapi_spec=openapi_spec,
    auth_type=AuthType.NO_AUTH
)
print(tool.info)

{'type': 'function', 'function': {'name': 'get_/search/titles/results/', 'description': 'Search newspaper titles', 'parameters': {'type': 'object', 'properties': {'terms': {'type': 'string', 'description': 'The search term or phrase for querying newspaper titles.'}, 'format': {'type': 'string', 'description': 'The format of the response.'}, 'page': {'type': 'integer', 'description': 'Page number for paginated results.'}}, 'required': ['terms', 'format', 'page']}}}


In this example, we're using AuthType.NO_AUTH because the Chronicling America API doesn't require authentication. For APIs that require authentication, you would use the appropriate AuthType and provide the necessary credentials.

## Execute an API call
To make an API call, create a ToolCall object with the necessary arguments and then execute it:


In [7]:
tool_call = ToolCall(
    name="get_historical_american_newspapers",
    arguments={
        "terms": "oakland",
        "format": "json",
        "page": 5
    }
)

result = tool.execute(tool_call)
print(result)

{'totalItems': 395, 'endIndex': 250, 'startIndex': 201, 'itemsPerPage': 20, 'items': [{'essay': [], 'place_of_publication': 'Oakland [Calif.]', 'start_year': 1854, 'publisher': 'R.B. Quayle & Co.', 'county': ['Alameda'], 'edition': None, 'frequency': 'Weekly', 'url': 'https://chroniclingamerica.loc.gov/lccn/sn88080878.json', 'id': '/lccn/sn88080878/', 'subject': ['Alameda County (Calif.)--Newspapers.', 'California--Alameda County.--fast--(OCoLC)fst01205577'], 'city': ['Oakland'], 'language': ['English'], 'title': 'Alameda County express.', 'holding_type': ['Microfilm Service Copy', 'Microfilm Service Copy', 'Original'], 'end_year': 1899, 'alt_title': [], 'note': ['Description based on: Vol. 1, no. 9 (May 13, 1854).'], 'lccn': 'sn88080878', 'state': ['California'], 'place': ['California--Alameda--Oakland'], 'country': 'California', 'type': 'title', 'title_normal': 'alameda county express.', 'oclc': '15533771'}, {'essay': [], 'place_of_publication': 'Oakland [Calif.]', 'start_year': 1859

This will make an API call to search for historical American newspapers with the term "oakland", requesting JSON format and the 5th page of results.


## Handling authentication
For APIs that require authentication, you would specify the auth_type when creating the `OpenAPITool` instance. For example:

```python
tool = OpenAPITool(
    name="some_api_tool",
    openapi_spec=some_api_spec,
    auth_type=AuthType.API_KEY,
    auth_key_prefix="MY_API"
)
```

In this case, the tool will look for environment variables like MY_API_API_KEY_NAME, MY_API_API_KEY_VALUE, and MY_API_API_KEY_IN to handle the authentication. We support API key with customized names in the following locations in  `header` or in request `params`. We support bearer token in the `header`, and basic authentication with `username` and `password`.